# Đi xuyên một ảnh thật — hiểu Mức 1

> **Notebook lịch sử — evaluator legacy.** Phần chấm bên dưới dùng đúng node ID, gộp GT trùng và pooled recall mỗi predicate. Kết quả không cùng protocol với báo cáo Level 1 hiện tại. Để học cách chấm reference GC/NG đang dùng, mở [level1_reference_demo.ipynb](level1_reference_demo.ipynb).

Mục tiêu: tự giải thích được **đầu vào → dự đoán → chấm điểm**, bằng một ảnh validation.
Chạy từng ô với **Shift+Enter** và đọc câu hỏi trước khi chạy ô tiếp theo.
Chọn kernel Python `myenv/bin/python` của dự án. Notebook chỉ suy luận CPU trên một ảnh.

Ba mô hình cùng seed 42 và cùng đặc trưng CLIP: MLP, GAT, GAT+WordNet.
Đây là minh họa cơ chế, không dùng một ảnh để kết luận mô hình nào tốt hơn trên toàn tập.

In [ ]:
from pathlib import Path
import sys
import json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/models/gat.py').exists()), None)
assert ROOT is not None, 'Mở notebook bên trong repository semantic-graph-editing'
sys.path.insert(0, str(ROOT))


In [ ]:
import torch
from IPython.display import display
from PIL import Image, ImageDraw
from scripts.train_mlp import make_dataset
from scripts.evaluate_level1 import load_model
from scripts.extract_level1_examples import predict_graph
from src.knowledge.transe.core import sha256
torch.set_num_threads(1)
DATA = ROOT / 'data/raw/vg150'
CACHE = ROOT / 'data/cache/vg150-clip-vit-b32-v1'
IMAGE_INDEX = 0  # Chỉ số trong validation, KHÔNG phải image_id. Đổi rồi chạy lại từ ô này.
plan = json.loads((ROOT / 'reports/level1/clip-all-27run-test-plan.json').read_text())
conditions = {'MLP': 'mlp_semantic', 'GAT': 'gat', 'GAT + WordNet': 'wordnet_kg_on'}
checkpoints, models = {}, {}
for name, condition in conditions.items():
    entry = next(e for e in plan['entries'] if e['condition'] == condition and e['seed'] == 42)
    path = ROOT / entry['checkpoint']
    assert sha256(path) == entry['checkpoint_sha256'], f'Checkpoint thay đổi: {path}'
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    config = checkpoint['config']
    assert config['data_sha256'] == plan['data_sha256']
    assert config['visual_cache_manifest_sha256'] == sha256(CACHE / 'manifest.json')
    assert config['box_policy'] == 'clip'
    checkpoints[name] = checkpoint
    models[name] = load_model(checkpoint).eval()
for filename, expected in plan['data_sha256'].items():
    assert sha256(DATA / filename) == expected, f'Dữ liệu khác: {filename}'
# Dataset kiểm tra nguồn cache CLIP và giữ đúng thứ tự object.
dataset = make_dataset(DATA, 'val', 'clip', CACHE)
graph = dataset[IMAGE_INDEX]
metadata = dataset.loader.image_metadata[dataset.loader.indices[IMAGE_INDEX]]
vocab = json.loads((DATA / 'VG-SGG-dicts-with-attri.json').read_text())
objects = {v:k for k,v in vocab['label_to_idx'].items()}
predicates = {v:k for k,v in vocab['predicate_to_idx'].items()}
image_path = DATA / 'images' / f'{graph.image_id}.jpg'
assert image_path.exists(), f'Thiếu JPEG: {image_path}'
print('Ảnh validation:', graph.image_id, '| số đối tượng:', graph.num_nodes)
print('Đã nạp 3 checkpoint; chưa train thêm.')

## 1. Nhìn ảnh: máy đã được cho những gì?

Ảnh dưới có **hộp và nhãn GT từ dataset**, không phải detector của chúng ta tìm ra.
Số trước nhãn là **node ID trong ảnh**. Hai người khác nhau có node ID khác nhau dù cùng nhãn.

**Tự hỏi:** Có những đối tượng nào? Bạn đoán quan hệ giữa chúng là gì?

In [ ]:
original = Image.open(image_path).convert('RGB')
def show_boxes(nodes):
    canvas = original.copy()
    draw = ImageDraw.Draw(canvas)
    for node in nodes:
        box = tuple(float(x) for x in graph.boxes[node])
        draw.rectangle(box, outline='red', width=3)
        label = f'{node}: {objects[int(graph.object_labels[node])]}'
        draw.text((box[0] + 2, box[1] + 2), label, fill='white', stroke_width=2, stroke_fill='black')
    canvas.thumbnail((1000, 700))
    display(canvas)
show_boxes(range(graph.num_nodes))
for node in range(graph.num_nodes):
    print(node, objects[int(graph.object_labels[node])], 'box XYXY:', graph.boxes[node].tolist())

## 2. Chọn một cặp và tự đoán trước

Mặc định lấy cặp của annotation đầu tiên để chắc chắn có đáp án đối chiếu, **không chọn theo model thắng/thua**.
Bạn có thể thay `SUBJECT` và `OBJECT` bằng node ID trong ảnh. Hướng rất quan trọng: người → ngựa khác ngựa → người.

**Trước khi chạy bước 4:** viết một quan hệ bạn đoán vào `MY_GUESS`.

In [ ]:
assert graph.gt_predicate.numel(), 'Ảnh không có GT relation; đổi IMAGE_INDEX'
SUBJECT = int(graph.gt_relation_index[0, 0])
OBJECT = int(graph.gt_relation_index[1, 0])
MY_GUESS = ''  # Ví dụ: on / near / riding; tự quan sát ảnh rồi điền.
assert SUBJECT != OBJECT and 0 <= SUBJECT < graph.num_nodes and 0 <= OBJECT < graph.num_nodes
show_boxes([SUBJECT, OBJECT])
print(f'{SUBJECT}: {objects[int(graph.object_labels[SUBJECT])]} → ? → {OBJECT}: {objects[int(graph.object_labels[OBJECT])]}')
print('Bạn đoán:', MY_GUESS or '(chưa điền)')

## 3. Biến đối tượng thành dãy số

Mỗi node có **5 số hình học + 512 số CLIP vùng ảnh** trong `graph.x`.
Nhãn được mô hình đổi thành embedding học được. GAT trao đổi thông tin giữa các node;
GAT+WordNet còn ghép vector KG cố định vào đầu vào.

`edge_index` là đường truyền tin **không chứa đáp án predicate**.
`edge_label_index` là các cặp phải dự đoán. Hai tên này không có nghĩa đã cho model biết quan hệ thật.

**Tự hỏi:** Tại sao chỉ biết nhãn “người” và “ngựa” chưa đủ phân biệt “cưỡi” với “đứng cạnh”?

In [ ]:
print('graph.x:', tuple(graph.x.shape), '= [số node, 5 + 512]')
print('Số cạnh truyền tin:', graph.edge_index.shape[1])
print('Số cặp ứng viên:', graph.edge_label_index.shape[1], '= N × (N − 1)')
kg_model = models['GAT + WordNet']
for node in [SUBJECT, OBJECT]:
    class_id = int(graph.object_labels[node])
    print('\nNode', node, objects[class_id])
    print('Hình học [cx, cy, w, h, area]:', graph.x[node, :5].tolist())
    print('CLIP:', graph.x[node, 5:].numel(), 'chiều')
    print('Embedding nhãn GAT:', models['GAT'].object_embedding.embedding_dim, 'chiều')
    print('KG:', kg_model.kg_features[class_id].numel(), 'chiều; mapping hợp lệ:', bool(kg_model.kg_valid[class_id]))
print('\nVector KG là số học được từ WordNet, không phải câu trả lời quan hệ của ảnh này.')

## 4. Cho ba mô hình dự đoán, rồi mở đáp án

Hàm suy luận chỉ nhận đặc trưng, nhãn đối tượng và các cạnh/cặp ứng viên.
GT predicate được đọc riêng để đối chiếu sau đó.

Bảng top-5 dưới đây là điểm softmax của **một cặp** (tính cả background trước khi chọn foreground).
Điểm không phải xác suất đúng đã được hiệu chuẩn. Evaluator chỉ chọn **một predicate foreground cao nhất mỗi cặp**.

In [ ]:
pairs = graph.edge_label_index
pair_id = torch.where((pairs[0] == SUBJECT) & (pairs[1] == OBJECT))[0].item()
results = {}
with torch.inference_mode():
    for name, model in models.items():
        logits = predict_graph(model, checkpoints[name]['config']['model'], graph, torch.device('cpu'))
        probabilities = logits.softmax(-1)
        scores, labels = probabilities[:, 1:].max(-1)
        labels += 1
        ranking = torch.argsort(scores, descending=True, stable=True)
        results[name] = (scores, labels, ranking)
        top_scores, top_ids = probabilities[pair_id, 1:].topk(5)
        print('\n' + name)
        for value, predicate_id in zip(top_scores.tolist(), (top_ids + 1).tolist(), strict=False):
            print(f'  {predicates[predicate_id]:20s} {value:.4f}')
gt = {(int(graph.gt_relation_index[0,i]), int(graph.gt_predicate[i]), int(graph.gt_relation_index[1,i]))
      for i in range(graph.gt_predicate.numel())}
answers = [predicates[p] for s,p,o in sorted(gt) if (s,o) == (SUBJECT,OBJECT)]
print('\nĐáp án dataset cho cặp đã chọn:', answers or '(không có annotation)')
print('Một cặp có thể có nhiều GT predicate. Thiếu annotation không chứng minh quan hệ không tồn tại.')

## 5. Từ một cặp đến R@50 của cả ảnh

Xếp hạng **mọi cặp trong ảnh** bằng điểm predicate đã chọn. Giữ tối đa 50 bộ ba.
Một GT được tìm lại nếu đúng **subject node, predicate và object node**, và nằm trong top-50.

Hai cách bỏ lỡ: chọn sai predicate, hoặc chọn đúng nhưng hạng thấp hơn 50.
Đây là **R@50 của một ảnh**, không phải mR@50 của toàn validation. mR@50 cần gom hits/support theo từng lớp quan hệ trên cả tập rồi lấy trung bình lớp.

In [ ]:
for name, (_scores, labels, ranking) in results.items():
    predicted = {(int(pairs[0,i]), int(labels[i]), int(pairs[1,i])) for i in ranking[:50].tolist()}
    hits = predicted & gt
    rank = int(torch.where(ranking == pair_id)[0].item()) + 1
    label = int(labels[pair_id])
    triplet = (SUBJECT, label, OBJECT)
    reason = 'Đúng predicate, trong top-50' if triplet in gt and rank <= 50 else (
        'Đúng predicate nhưng ngoài top-50' if triplet in gt else 'Không khớp GT của cặp')
    print(f'\n{name}: R@50 ảnh = {len(hits)}/{len(gt)} = {len(hits)/len(gt):.1%}')
    print(f'Cặp đang xem: {predicates[label]}, hạng {rank}, {reason}')
    print('Các GT được tìm lại:')
    for s,p,o in sorted(hits):
        print(f'  {s}:{objects[int(graph.object_labels[s])]} → {predicates[p]} → {o}:{objects[int(graph.object_labels[o])]}')

## 6. Tự giải thích lại bằng lời

Điền câu trả lời sau khi quan sát, không chỉ chép metric:

1. Ảnh này đã cho mô hình những thông tin nào?
2. Bạn chọn cặp nào, tự đoán gì, GT ghi gì?
3. MLP/GAT/GAT+WordNet dự đoán khác nhau thế nào?
4. Nếu bỏ lỡ GT: sai predicate hay đúng nhưng ngoài top-50?
5. Vì sao một ảnh chưa đủ chứng minh KG hữu ích hoặc vô ích?

**Tiếp theo:** đổi cặp trong cùng ảnh, chạy lại từ bước 2. Sau đó đổi `IMAGE_INDEX` và chạy lại từ đầu.
Đây là ba checkpoint của ma trận CLIP ban đầu, chưa dùng loss điều chỉnh/fusion mới; không gán metric 25–26% của nhánh mới cho chúng.

**Một điểm đáng quan sát ở ảnh mặc định:** `wears` và `wearing` là hai predicate ID khác nhau. Dự đoán gần nghĩa vẫn có thể bị chấm không khớp; evaluator không tự gộp từ đồng nghĩa. Hãy phân biệt lỗi theo nhãn benchmark với nhận xét ngữ nghĩa của bạn.